In [ ]:
# Código para extraer el json de:
# https://datos.madrid.es/egob/catalogo/206974-0-agenda-eventos-culturales-100.json

# 

In [1]:
import os
from bs4 import BeautifulSoup
import requests

# 1. Configuración de la URL y el contenedor
URL = "https://www.madrid.es/sites/v/index.jsp?vgnextchannel=ca9671ee4a9eb410VgnVCM100000171f5a0aRCRD&vgnextoid=0b6fae21a7cbb910VgnVCM100000891ecb1aRCRD"
ID_O_CLASE_DEL_DIV = "image-content ic-right"  # Cambia esto
URL_BASE = "https://www.madrid.es"  # Cambia esto si la URL base es diferente

def extraer_imagen(URL, ID_O_CLASE_DEL_DIV, URL_BASE):
    # 2. Descargar el HTML de la página
    headers = {"User-Agent": "Mozilla/5.0"}
    respuesta = requests.get(URL)

    if respuesta.status_code != 200:
        print(f"Error al acceder a la página: {respuesta.status_code}")
        return

    # 3. Parsear el HTML
    soup = BeautifulSoup(respuesta.text, "html.parser")

    # 4. Buscar el div específico (Elige Opción A o Opción B)
    # Opción A: Si el div tiene una clase CSS específica
    contenedor = soup.find("div", class_=ID_O_CLASE_DEL_DIV)

    # Opción B: Si el div tiene un ID único (descomenta la línea de abajo)
    # contenedor = soup.find('div', id=ID_O_CLASE_DEL_DIV)

    if not contenedor:
        print("No se encontró el div especificado.")
        return

    # 5. Extraer la etiqueta de la imagen dentro de ese div
    etiqueta_img = contenedor.find("img")

    if not etiqueta_img:
        print("No se encontró ninguna imagen dentro del div.")
        return

    # 6. Obtener la URL de la imagen (atributo 'src')
    url_imagen = etiqueta_img.get("src")

    # Asegurar que la URL sea completa si es relativa
    if url_imagen and not url_imagen.startswith(("http://", "https://")):
        from urllib.parse import urljoin  # Python 3

        url_imagen = urljoin(URL_BASE, url_imagen)

    print(f"URL de la imagen encontrada: {url_imagen}")
    return url_imagen


# Ejecutar la función
extraer_imagen(URL, ID_O_CLASE_DEL_DIV, URL_BASE)


URL de la imagen encontrada: https://www.madrid.es/UnidadesDescentralizadas/Bibliotecas/BibliotecasPublicas/Actividades/Exposiciones/ficheros/251001cuestamoyano_260.png


'https://www.madrid.es/UnidadesDescentralizadas/Bibliotecas/BibliotecasPublicas/Actividades/Exposiciones/ficheros/251001cuestamoyano_260.png'

In [23]:
from datetime import datetime
import json
import random


# Funciones auxiliares simuladas (debes adaptarlas a tu lógica real)
def extract_category(item_type):
    return item_type


def extract_district(district_id):
    return district_id


def populate_filters():
    pass


def apply_filters():
    pass


# Variable global para almacenar los resultados
all_activities = []


def load_activities_local(file_path="actividades.json"):
    global all_activities

    try:
        # Abre y lee el archivo JSON local
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)

        graph = data.get("@graph")

        if isinstance(graph, list):
            temp_activities = []

            for item in graph:
                # Extrae datos de diccionarios anidados con seguridad
                address = item.get("address", {})
                district_obj = address.get("district", {}) if address else {}
                area = address.get("area", {}) if address else {}
                location_obj = item.get("location", {})

                # Manejo y conversión de fechas
                dtstart = item.get("dtstart")
                date_val = datetime.fromisoformat(dtstart) if dtstart else None

                dtend = item.get("dtend")
                end_date_val = datetime.fromisoformat(dtend) if dtend else None

                # Conversión de coordenadas
                try:
                    lat = (
                        float(location_obj.get("latitude"))
                        if location_obj
                        else None
                    )
                    lon = (
                        float(location_obj.get("longitude"))
                        if location_obj
                        else None
                    )
                except (ValueError, TypeError):
                    lat, lon = None, None

                # Construcción del diccionario
                activity = {
                    "id": item.get("@id", str(random.random())),
                    "title": item.get("title", "Sin título"),
                    "description": item.get("description", ""),
                    "category": extract_category(item.get("@type")),
                    "location": item.get("event-location", ""),
                    "district": extract_district(district_obj.get("@id")),
                    "lat": lat,
                    "lon": lon,
                    "date": date_val,
                    "endDate": end_date_val,
                    "time": item.get("time", ""),
                    "free": item.get("free") == 1 or item.get("free") is True,
                    "price": item.get("price", ""),
                    "audience": item.get("audience", ""),
                    "link": item.get("link", ""),
                    "street": area.get("street-address", "") if area else "",
                }
                temp_activities.append(activity)

            # Filtrar actividades pasadas
            today = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
            all_activities = [
                a for a in temp_activities if a["date"] is None or a["date"] >= today
            ]

            # Ejecutar funciones posteriores
            populate_filters()
            apply_filters()

            print(
                f"Datos cargados desde {file_path} y filtrados con éxito. Total: {len(all_activities)}"
            )
            return all_activities

        else:
            print("Error: formato inesperado (Falta '@graph')")

    except FileNotFoundError:
        print(f"Error: No se encontró el archivo en la ruta {file_path}")
    except json.JSONDecodeError:
        print("Error: El archivo no es un JSON válido")
    except Exception as error:
        print(f"Error inesperado: {error}")


# Ejecución del código síncrono
if __name__ == "__main__":
    aclist = load_activities_local("actividades.json")


Datos cargados desde actividades.json y filtrados con éxito. Total: 651


In [24]:
aclist


[{'id': 'https://datos.madrid.es/egob/catalogo/tipo/evento/12008124-10-festival-internacional-folclore-latina-folkfestival-ara-madrid-.json',
  'title': "10º Festival Internacional de Folclore de Latina.Folkfestival 'Ara de Madrid'",
  'description': "10º Festival Internacional de Folclore de Latina – Folkfestival 'Ara de Madrid', con la participación de los grupos:  Grupo Folclórico 'Villaviciosa-Aires de Asturias' (Asturias) bajo la dirección de Elena Alonso.  Grupo 'Leyendas de México' (México), bajo la dirección de Ana Paola de la Cruz.  Ballet 'Ara de Madrid', bajo la dirección de Carmina Villar.",
  'category': 'https://datos.madrid.es/egob/kos/actividades/Fiestas',
  'location': '',
  'district': None,
  'lat': None,
  'lon': None,
  'date': datetime.datetime(2026, 6, 7, 19, 0),
  'endDate': datetime.datetime(2026, 6, 7, 23, 59),
  'time': '19:00',
  'free': True,
  'price': '',
  'audience': '',
  'link': 'http://www.madrid.es/sites/v/index.jsp?vgnextchannel=ca9671ee4a9eb410Vgn

In [ ]:
# For each activity, get the link and extract the image URL using the previous function,
#  then add it to the activity dictionary under the key "image_url".
all_activities_image = []
i = 0
for activity in all_activities:
    # Aquí debes implementar la lógica para obtener el enlace de la actividad
    # Por ejemplo, podrías usar activity['link'] si ese campo contiene la URL
    activity_link = activity.get("link")
    # Datetime a texto ISO para JSON:
    activity["date"] = activity["date"].isoformat() if activity["date"] else None
    activity["endDate"] = activity["endDate"].isoformat() if activity["endDate"] else None

    if activity_link:
        # Función extraer_imagen() con URL
        try:
            image_url = extraer_imagen(activity_link, ID_O_CLASE_DEL_DIV, URL_BASE)
            activity["image_url"] = image_url
            i += 1
        except Exception as e:
            print(f"Error al extraer imagen {activity.get('name')}: {e}")
            activity["image_url"] = None
    else:
        activity["image_url"] = None  # O algún valor por defecto si no hay enlace

    all_activities_image.append(activity)

    # Paramos si llega a 10 imagenes:
    if i >= 10:
        print("Se han extraído 10 imágenes, deteniendo el proceso.")
        break

# Almacenamos archivo JSON con las actividades y sus imágenes
# with open("actividades_con_imagenes.json", "w", encoding="utf-8") as f:
#     json.dump(all_activities_image, f, ensure_ascii=False, indent=4)

URL de la imagen encontrada: https://www.madrid.es/UnidadesDescentralizadas/DistritoLatina/Actividades/Fiestas%20de%20Aluche/7_7_Ballet_Ara.jpg
No se encontró el div especificado.
URL de la imagen encontrada: https://www.madrid.es/UnidadesDescentralizadas/CiudadDistrito/ficheros/CIUDAD_DISTRITO_233150_es.jpeg
URL de la imagen encontrada: https://www.madrid.es/UnidadesDescentralizadas/DistritoRetiro/FICHEROS/FICHEROS%20ACTIVIDADES%20MAYO/30%20may%20CV%20FIGUE%202026_6%20x%204%20&%20CUEROS.jpg
URL de la imagen encontrada: https://www.madrid.es/UnidadWeb/UGBBDD/MadridDestino/Eventos/ficheros/MADRID_DESTINO_188437_es.jpeg
URL de la imagen encontrada: https://www.madrid.es/UnidadesDescentralizadas/DistritoLatina/Actividades/1_FICHEROS%202026/Ficheros_mayo_26/SM_25_5_TERSA_BERGANZA.jpg
URL de la imagen encontrada: https://www.madrid.es/UnidadWeb/UGBBDD/Actividades/Distritos/PuenteVallecas/Actividades/ficheros/Mayo26/2705Agrucha.jpg
URL de la imagen encontrada: https://www.madrid.es/UnidadesD

In [28]:
all_activities_image_dict

{'https://datos.madrid.es/egob/catalogo/tipo/evento/12008124-10-festival-internacional-folclore-latina-folkfestival-ara-madrid-.json': {'id': 'https://datos.madrid.es/egob/catalogo/tipo/evento/12008124-10-festival-internacional-folclore-latina-folkfestival-ara-madrid-.json',
  'title': "10º Festival Internacional de Folclore de Latina.Folkfestival 'Ara de Madrid'",
  'description': "10º Festival Internacional de Folclore de Latina – Folkfestival 'Ara de Madrid', con la participación de los grupos:  Grupo Folclórico 'Villaviciosa-Aires de Asturias' (Asturias) bajo la dirección de Elena Alonso.  Grupo 'Leyendas de México' (México), bajo la dirección de Ana Paola de la Cruz.  Ballet 'Ara de Madrid', bajo la dirección de Carmina Villar.",
  'category': 'https://datos.madrid.es/egob/kos/actividades/Fiestas',
  'location': '',
  'district': None,
  'lat': None,
  'lon': None,
  'date': '2026-06-07T19:00:00',
  'endDate': '2026-06-07T23:59:00',
  'time': '19:00',
  'free': True,
  'price': ''

In [27]:
# Convertimos la lista de actividades a diccionario:
all_activities_image_dict = {activity.get('id'): activity for i, activity in enumerate(all_activities_image)}
all_activities_image_dict

# Ahora a JSON y la guardamos en un archivo
with open("actividades_con_imagenes.json", "w", encoding="utf-8") as f:
    json.dump(all_activities_image_dict, f, ensure_ascii=False, indent=4)

In [ ]:
# Subir json a firebase storage:



# Firebase subida

In [33]:
%pip install firebase-admin python-dotenv pyrebase4


  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 19.5 MB/s  0:00:00
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)

  Attempting uninstall: urllib3

    Found existing installation: urllib3 2.6.3

    Uninstalling urllib3-2.6.3:

      Successfully uninstalled urllib3-2.6.3

   ---------------------------------------- 0/9 [urllib3]
   ---------------------------------------- 0/9 [urllib3]
   ---------------------------------------- 0/9 [urllib3]
   ---------------------------------------- 0/9 [urllib3]
   ---------------------------------------- 0/9 [urllib3]
   ---------------------------------------- 0/9 [urllib3]
   ---------------------------------------- 0/9 [urllib3]
   ---- ----------------------------------- 1/9 [rsa]
   ---- ----------------------------------- 1/9 [rsa]
   ---- -------------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 2.10.1 requires protobuf<5,>=3.12.0, but you have protobuf 6.33.6 which is incompatible.


In [35]:
import os
import pyrebase
from dotenv import load_dotenv

# 1. Cargar las variables de entorno
load_dotenv()

# 2. Estructurar la configuración mapeando tus variables
config = {
    "apiKey": os.getenv("FIREBASE_apiKey"),
    "authDomain": os.getenv("FIREBASE_authDomain"),
    "projectId": os.getenv("FIREBASE_projectId"),
    "storageBucket": os.getenv("FIREBASE_storageBucket"),
    "messagingSenderId": os.getenv("FIREBASE_messagingSenderId"),
    "appId": os.getenv("FIREBASE_appId"),
    "databaseURL": ""  # Se puede dejar vacío si solo usas Storage
}

# 3. Inicializar Firebase y el servicio de Storage
firebase = pyrebase.initialize_app(config)
storage = firebase.storage()

# 4. Definir rutas de los archivos
ruta_archivo_local = "actividades_con_imagenes.json"
ruta_destino_en_nube = "actividades_con_imagenes.json"

# 5. Subir el archivo
# En pyrebase se usa child() para la ruta destino y put() para el archivo local
storage.child(ruta_destino_en_nube).put(ruta_archivo_local)

print("¡Archivo subido con éxito usando las credenciales del .env!")

¡Archivo subido con éxito usando las credenciales del .env!
